We use rate and credit spread to our model:
Does rate and credit spread explain BDC pricing?

In [29]:
# import data for regression analysis

import pandas as pd

df_reg = pd.read_csv("data/df_event_study_stg.csv", parse_dates=["announcement_date","trade_date"])
df_rate = pd.read_csv("data/df_rate.csv", parse_dates=["rate_date"])


In [30]:
# merge trade and rate 

import pandas as pd

df_reg = df_reg.copy()
df_rate = df_rate.copy()

# Parse dates
df_reg.loc[:, "trade_date"] = pd.to_datetime(df_reg["trade_date"])
df_reg.loc[:, "announcement_date"] = pd.to_datetime(df_reg["announcement_date"])
df_rate.loc[:, "rate_date"] = pd.to_datetime(df_rate["rate_date"])

# Sort
df_reg = df_reg.sort_values(["ticker", "trade_date"]).copy()
df_rate = df_rate.sort_values("rate_date").copy()

# Fill macro levels first
rate_cols = ["dgs10", "dbaa", "daaa", "spread"]
df_rate.loc[:, rate_cols] = df_rate.loc[:, rate_cols].ffill()

# Build daily changes
df_rate.loc[:, "d_dgs10"] = df_rate["dgs10"].diff()
df_rate.loc[:, "d_spread"] = df_rate["spread"].diff()

# Remove any old versions before merging again
cols_to_drop = [
    "dgs10", "spread", "d_dgs10", "d_spread",
    "nav_x_dgs10", "nav_x_spread", "rate_date"
]
df_reg = df_reg.drop(columns=[c for c in cols_to_drop if c in df_reg.columns], errors="ignore")

# Merge
df_reg = df_reg.merge(
    df_rate.loc[:, ["rate_date", "dgs10", "spread", "d_dgs10", "d_spread"]],
    left_on="trade_date",
    right_on="rate_date",
    how="left"
)

df_reg = df_reg.drop(columns=["rate_date"])

# Interactions
df_reg.loc[:, "nav_x_dgs10"] = df_reg["nav_ret"] * df_reg["d_dgs10"]
df_reg.loc[:, "nav_x_spread"] = df_reg["nav_ret"] * df_reg["d_spread"]

df_reg.head()


,ticker,announcement_date,trade_date,nav,nav_ret,ret,price,excess_ret,dgs10,spread,d_dgs10,d_spread,nav_x_dgs10,nav_x_spread
0,ARCC,2015-11-04,2015-12-01,16.789579,-0.0005,0.003161,15.87,-0.006347,2.15,3.21,NaN,NaN,NaN,NaN
1,ARCC,2015-11-04,2015-12-02,16.789579,-0.0005,-0.008822,15.73,0.001900,2.18,3.17,0.03,-0.04,-0.000015,0.000020
2,ARCC,2015-11-04,2015-12-03,16.789579,-0.0005,-0.003179,15.68,0.011138,2.33,3.18,0.15,0.01,-0.000075,-0.000005
3,ARCC,2015-11-04,2015-12-04,16.789579,-0.0005,0.005102,15.76,-0.011352,2.28,3.15,-0.05,-0.03,0.000025,0.000015
4,ARCC,2015-11-04,2015-12-07,16.789579,-0.0005,-0.025381,15.36,-0.015612,2.23,3.15,-0.05,0.00,0.000025,-0.000000


In [32]:
df_reg.to_csv("data/df_reg_rate_event_study.csv", index=False)

In [40]:

cols_needed = [
    "ticker",
    "announcement_date",
    "trade_date",
    "ret",
    "nav_ret",
    "dgs10",
    "spread",
    "d_dgs10",
    "d_spread",
    "nav_x_dgs10",
    "nav_x_spread"
]

df = (
    df_reg.loc[:, cols_needed]
    .loc[
        lambda d: (d["trade_date"] >= "2016-01-01") &
                  (d["trade_date"] <= "2024-12-31")
    ])

df.to_csv("data/df_reg_rate_event_study_clean.csv", index=False)

In [39]:
import statsmodels.api as sm

X = df[[
    "nav_ret",
    "d_dgs10",
    "d_spread",
    "nav_x_dgs10",
    "nav_x_spread"
]].astype(float)

X = sm.add_constant(X)
y = df["ret"].astype(float)

model = sm.OLS(y, X).fit()
print(model.summary())


                            OLS Regression Results                            
Dep. Variable:                    ret   R-squared:                       0.072
Model:                            OLS   Adj. R-squared:                  0.071
Method:                 Least Squares   F-statistic:                     70.69
Date:                Wed, 15 Apr 2026   Prob (F-statistic):           2.00e-71
Time:                        18:38:43   Log-Likelihood:                 12273.
No. Observations:                4528   AIC:                        -2.453e+04
Df Residuals:                    4522   BIC:                        -2.449e+04
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
const            0.0006      0.000      2.213   

Interpretations:


nav:
On its own, lagged NAV return does not significantly explain daily BDC returns.
That supports the idea that stale NAV is not enough by itself to explain market pricing.

d_spread:
When credit spreads widen, BDC returns fall.
This is your clearest macro result.
It says investors are sensitive to changing credit conditions and risk premia.

nav_x_dgs10:
This is the strongest interaction result.
The negative coefficient says that when Treasury yields move, the relationship between lagged NAV and returns becomes weaker.
That fits your thesis that discount-rate shocks make investors rely less on stale fundamentals.

nav_x_spread:
This is also significant, though smaller and less strong than the Treasury interaction.
It says the pricing impact of NAV changes when credit conditions move.
That supports the broader claim that NAV’s relevance is state-dependent.

Summary:

Lagged NAV by itself is not significant.
Credit spread changes have a strong direct negative effect on BDC returns.
More importantly, the effect of NAV is conditional on macro conditions.
When Treasury yields move, NAV matters less for returns.
This supports our thesis that investors shift attention away from stale fundamentals when discount rates and risk premia are moving.

Conclusion

1. The Treasury interaction strongly supports the hypothesis that NAV’s pricing relevance weakens when discount-rate conditions shift.
2. The spread interaction is also significant, indicating that the NAV-return relationship changes with credit conditions, though the sign is more nuanced.